# 05 · Templated TS — a known geometry onto a fresh molecule

Take a **known TS geometry** and transfer its reacting core onto a *different* molecule. `template=` is the TS (an `.xyz` or an `Ensemble`); `match=` a SMARTS for the core; `anchor=` maps the SMARTS labels to the template's atoms. The matched core is pinned to the template exactly (rigid graft) and everything else is conformer-searched — **one TS becomes a whole analogue series**. The overlay renders make the point obvious: all conformers aligned on the pinned core show it held rigid while the periphery fans out.

In [ ]:
import itertools, tempfile
import numpy as np
import rxembed as rx
import xyzrender
from rdkit import Chem, RDLogger
from rxembed.embed.dispatch import _xyz_to_mol

RDLogger.DisableLog("rdApp.*")
rx.set_verbose("INFO")
SN2 = "structures/sn2.xyz"  # a known SN2 TS geometry (F⁻ + R–Cl)
CORE = "[F-:1].[#6:2]-[Cl:3]"  # the reacting core, atoms labelled 1/2/3
ANCHOR = {1: 4, 2: 0, 3: 5}  # label -> template atom index (F=4, C=0, Cl=5)


def core_dev(ens):
    "Max deviation (Å) of the matched F···C···Cl core from the template geometry (0 = pinned exactly)."
    tp = _xyz_to_mol(SN2, 0).GetConformer().GetPositions()
    nm = ens.mol.GetSubstructMatch(Chem.MolFromSmarts("[F-].[#6]-[Cl]"))
    idx = [4, 0, 5]
    pairs = list(itertools.combinations(range(3), 2))
    ref = [np.linalg.norm(tp[idx[i]] - tp[idx[j]]) for i, j in pairs]
    d = lambda q, a, b: float(np.linalg.norm(q[nm[a]] - q[nm[b]]))
    return max(
        max(abs(d(ens.mol.GetConformer(c).GetPositions(), i, j) - r) for (i, j), r in zip(pairs, ref)) for c in ens.ids
    )

## One fresh substrate onto the SN2 core

1-chloropropane + F⁻: the SMARTS core is matched on the *new* molecule and pinned to the template's F···C···Cl geometry (dev ≈ 0); the propyl group is searched. The forming/breaking bonds are drawn dashed.

In [ ]:
ens = rx.embed("[F-].CCCCl", template=SN2, match=CORE, anchor=ANCHOR).mc().prune()
print(f"1-chloropropane + F⁻: {len(ens)} conformers | [F···C···Cl] core pinned to {core_dev(ens):.4f} Å")
nm = ens.mol.GetSubstructMatch(Chem.MolFromSmarts("[F-].[#6]-[Cl]"))  # (F, C, Cl) in the new molecule
xyzrender.render(
    ens.lowest(1).dump(tempfile.mktemp(suffix=".xyz")),
    no_hy=True,
    ts_bonds=[(nm[0] + 1, nm[1] + 1), (nm[1] + 1, nm[2] + 1)],
)

The **overlay** makes the transfer unmistakable — every conformer aligned on the F···C···Cl core: it sits rigid while the propyl tail fans out (the part actually being searched).

In [ ]:
p = ens.align(on="[F-].[#6]-[Cl]").dump(tempfile.mktemp(suffix=".xyz"))
xyzrender.render(xyzrender.load(p, ensemble=True), no_hy=True)

## The point — an analogue *series* on one core

Transfer the same TS onto isopropyl, hexyl, and benzyl chlorides — each keeps the reacting core pinned (dev ≈ 0) while its own substituent is searched.

In [ ]:
for sub in ["[F-].CC(C)Cl", "[F-].CCCCCCCl", "[F-].c1ccccc1CCl"]:
    e = rx.embed(sub, template=SN2, match=CORE, anchor=ANCHOR).mc(preset="rapid").prune()
    print(f"  {sub:18s}: {len(e):2d} confs | core dev {core_dev(e):.4f} Å")

## A template needn't be a TS — an `Ensemble` works

Any geometry can be a template. Embed a benzaldehyde scaffold, then graft that 8-atom aromatic-aldehyde core onto a fresh *ortho*-alkoxy substrate. The overlay (aligned on the benzaldehyde core) shows the core held rigid with the appended `–OCCCC` chain sweeping around it — exactly what the template pinned vs searched.

In [ ]:
scaffold = rx.embed("O=Cc1ccccc1").mc().prune()  # the template geometry (an Ensemble)
ens = rx.embed("O=Cc1ccccc1OCCCC", template=scaffold, match="O=Cc1ccccc1").mc().prune()
q = Chem.MolFromSmarts("O=Cc1ccccc1")
nm, tm = ens.mol.GetSubstructMatch(q), scaffold.mol.GetSubstructMatch(q)
tp = scaffold.mol.GetConformer(scaffold.ids[0]).GetPositions()
pw = lambda p_, ix: sorted(np.linalg.norm(p_[a] - p_[b]) for i, a in enumerate(ix) for b in ix[i + 1 :])
dev = max(max(abs(x - y) for x, y in zip(pw(ens.mol.GetConformer(c).GetPositions(), nm), pw(tp, tm))) for c in ens.ids)
print(f"benzaldehyde core templated onto a fresh substrate: {len(ens)} confs | 8-atom core dev {dev:.4f} Å")
p = ens.align(on="O=Cc1ccccc1").dump(tempfile.mktemp(suffix=".xyz"))
xyzrender.render(xyzrender.load(p, ensemble=True), no_hy=True)  # core fixed; -OCCCC tail fans out